# Weighted Least Squares Regression

**Topics:** Heteroscedasticity, Weighted Regression, Robust Estimation

## Overview

This notebook demonstrates weighted least squares (WLS) regression for data with heteroscedasticity (non-constant variance). When variance changes across observations, weighted regression provides more efficient estimates than ordinary least squares.

## What You'll Learn

- Recognize heteroscedasticity in data
- Understand the problem with OLS under heteroscedasticity
- Fit weighted regression models
- Determine appropriate weights
- Compare OLS vs WLS performance
- Validate heteroscedasticity corrections

---

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from aurora.models import fit_glm
from aurora.inference.diagnostics import glm_diagnostics
from scipy import stats

sns.set_style('whitegrid')
np.random.seed(42)

## 2. What is Heteroscedasticity?

**Homoscedasticity** (constant variance):
- Var(ε) = σ² (same for all observations)
- Standard assumption in OLS

**Heteroscedasticity** (non-constant variance):
- Var(ε) = σᵢ² (varies across observations)
- Common in real data!

**Examples where variance increases:**
- Investment returns vs portfolio size
- Expenditure variance vs income level
- Measurement error vs signal magnitude

**Problem:** OLS is still unbiased but:
- Inefficient (larger standard errors than necessary)
- Confidence intervals and p-values incorrect

**Solution:** Weighted Least Squares (WLS)

## 3. Generate Heteroscedastic Data

Simulate company revenue vs advertising spend where variance increases with spending:

In [ ]:
n = 150

# Advertising spend (in $1000s)
advertising = np.random.uniform(10, 100, n)

# True relationship: revenue = 50 + 2 * advertising
true_revenue = 50 + 2 * advertising

# Heteroscedastic noise: SD proportional to advertising
# Higher spending → more variability in returns
noise_sd = 0.3 * advertising  # SD increases with X
noise = np.random.randn(n) * noise_sd
revenue = true_revenue + noise

# Create DataFrame
df = pd.DataFrame({
    'advertising': advertising,
    'revenue': revenue,
    'true_revenue': true_revenue,
    'noise_sd': noise_sd
})

print(f"Generated data for {n} companies")
print(f"\nAdvertising spend range: ${df['advertising'].min():.1f}k - ${df['advertising'].max():.1f}k")
print(f"Revenue range: ${df['revenue'].min():.1f}k - ${df['revenue'].max():.1f}k")

## 4. Visualize Heteroscedasticity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Scatter plot showing increasing variance
axes[0].scatter(df['advertising'], df['revenue'], alpha=0.6, edgecolor='k', linewidth=0.5)
axes[0].plot(df['advertising'], df['true_revenue'], 'r-', lw=2, label='True relationship')

# Add visual guide for variance envelope
sorted_idx = np.argsort(df['advertising'])
x_sorted = df['advertising'].values[sorted_idx]
true_sorted = df['true_revenue'].values[sorted_idx]
sd_sorted = df['noise_sd'].values[sorted_idx]
axes[0].fill_between(x_sorted, true_sorted - 2*sd_sorted, true_sorted + 2*sd_sorted,
                      alpha=0.2, color='red', label='±2 SD')

axes[0].set_xlabel('Advertising Spend ($1000s)')
axes[0].set_ylabel('Revenue ($1000s)')
axes[0].set_title('Heteroscedastic Data\n(Variance increases with X)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Variance by bins
df['advertising_bin'] = pd.cut(df['advertising'], bins=5, labels=['Very Low', 'Low', 'Med', 'High', 'Very High'])
residuals_temp = df['revenue'] - df['true_revenue']
var_by_bin = df.groupby('advertising_bin', observed=True).agg({
    'advertising': 'mean',
    'revenue': lambda x: np.var(x - (50 + 2*df.loc[x.index, 'advertising']))
}).rename(columns={'revenue': 'variance'})

axes[1].bar(range(len(var_by_bin)), var_by_bin['variance'], color='steelblue', alpha=0.7, edgecolor='k')
axes[1].set_xticks(range(len(var_by_bin)))
axes[1].set_xticklabels(var_by_bin.index, rotation=45)
axes[1].set_ylabel('Residual Variance')
axes[1].set_xlabel('Advertising Spend Level')
axes[1].set_title('Variance by Spending Level\n(Clear heteroscedasticity!)')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nVariance ratio (High/Low): {var_by_bin['variance'].iloc[-1] / var_by_bin['variance'].iloc[0]:.1f}x")
print(f" Variance at high spending is much larger than at low spending!")

## 5. Fit OLS (Unweighted Regression)

First, let's try standard OLS and see what goes wrong:

In [ ]:
# Design matrix (just the advertising column, let fit_glm add intercept)
X = df['advertising'].values.reshape(-1, 1)
y = df['revenue'].values

# Fit OLS
result_ols = fit_glm(X=X, y=y, family='gaussian')

print("OLS Results (Ignoring Heteroscedasticity):")
print(result_ols.summary())
print("\n" + "="*60)

## 6. Diagnose Heteroscedasticity

Check residual plots:

In [ ]:
# Get residuals and fitted values
diagnostics_ols = glm_diagnostics(result_ols)
residuals_ols = diagnostics_ols.response_residuals
fitted_ols = result_ols.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Fitted (funnel shape = heteroscedasticity)
axes[0].scatter(fitted_ols, residuals_ols, alpha=0.6, edgecolor='k', linewidth=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('OLS: Residuals vs Fitted\n⚠️ Funnel shape indicates heteroscedasticity')
axes[0].grid(alpha=0.3)

# Scale-Location (should be flat if homoscedastic)
sqrt_abs_resid = np.sqrt(np.abs(diagnostics_ols.studentized_residuals))
axes[1].scatter(fitted_ols, sqrt_abs_resid, alpha=0.6, edgecolor='k', linewidth=0.5)

# Add trend line to highlight pattern
from scipy.interpolate import make_interp_spline
sorted_idx = np.argsort(fitted_ols)
spl = make_interp_spline(fitted_ols[sorted_idx], sqrt_abs_resid[sorted_idx], k=3)
x_smooth = np.linspace(fitted_ols.min(), fitted_ols.max(), 100)
y_smooth = spl(x_smooth)
axes[1].plot(x_smooth, y_smooth, 'r-', lw=2, label='Trend')

axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('√|Standardized Residuals|')
axes[1].set_title('OLS: Scale-Location Plot\n⚠️ Upward trend confirms heteroscedasticity')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Breusch-Pagan test for heteroscedasticity
# Test if squared residuals depend on X
resid_sq = residuals_ols**2
X_test = X[:, 1:2]  # Just the advertising column
X_test_full = np.column_stack([np.ones(n), X_test])
aux_result = fit_glm(X=X_test_full, y=resid_sq, family='gaussian')

# Calculate R² manually: R² = 1 - (SSE / SST)
y_pred_aux = aux_result.predict(X_test_full)
sse = np.sum((resid_sq - y_pred_aux)**2)
sst = np.sum((resid_sq - np.mean(resid_sq))**2)
r2_aux = 1 - (sse / sst) if sst > 0 else 0

bp_stat = (n - X_test_full.shape[1]) * r2_aux
bp_pval = 1 - stats.chi2.cdf(bp_stat, df=X_test.shape[1])

print(f"\nBreusch-Pagan Test for Heteroscedasticity:")
print(f"  H0: Homoscedasticity (constant variance)")
print(f"  Test statistic: {bp_stat:.4f}")
print(f"  p-value: {bp_pval:.6f}")
print(f"\n  Conclusion: {'Reject H0' if bp_pval < 0.05 else 'Fail to reject H0'} at 5% level")
print(f"  {'Heteroscedasticity detected!' if bp_pval < 0.05 else 'No heteroscedasticity detected'}")

## 7. Fit Weighted Least Squares (WLS)

### Determining Weights

If we know that Var(εᵢ) = σ²·xᵢ, then use weights:
- wᵢ = 1/xᵢ

More generally:
- If Var(εᵢ) ∝ xᵢᵖ, use wᵢ = 1/xᵢᵖ

In our case, SD ∝ x, so Var ∝ x², thus wᵢ = 1/xᵢ²

In [ ]:
# Compute weights based on known variance structure
# Var(ε) ∝ advertising², so weights = 1/advertising²
weights = 1 / (df['advertising']**2)

# Normalize weights (optional, for interpretation)
weights = weights / weights.mean()

# Fit WLS (X should be the same as OLS)
result_wls = fit_glm(
    X=X,
    y=y,
    family='gaussian',
    weights=weights
)

print("WLS Results (Accounting for Heteroscedasticity):")
print(result_wls.summary())
print("\n" + "="*60)

## 8. Compare OLS vs WLS

### Coefficient Estimates

In [ ]:
coef_names = ['Intercept', 'Advertising']
true_coefs = [50, 2]

# Combine intercept and coefficients
ols_coefs = np.concatenate([[result_ols.intercept_], result_ols.coef_])
wls_coefs = np.concatenate([[result_wls.intercept_], result_wls.coef_])

# Get standard errors (including intercept)
ols_se = np.concatenate([[result_ols.intercept_std_error_], result_ols.std_errors_])
wls_se = np.concatenate([[result_wls.intercept_std_error_], result_wls.std_errors_])

comparison_df = pd.DataFrame({
    'Coefficient': coef_names,
    'True': true_coefs,
    'OLS': ols_coefs,
    'WLS': wls_coefs,
    'OLS_SE': ols_se,
    'WLS_SE': wls_se
})
comparison_df['SE_Ratio'] = comparison_df['OLS_SE'] / comparison_df['WLS_SE']

print("\nCoefficient Comparison:")
print(comparison_df.to_string(index=False))
print(f"\n Both methods estimate coefficients well (unbiased)")
print(f" WLS has smaller standard errors (more efficient!)")
print(f" Average SE reduction: {(1 - comparison_df['WLS_SE'].mean()/comparison_df['OLS_SE'].mean())*100:.1f}%")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(coef_names))
width = 0.25

ax.bar(x - width, true_coefs, width, label='True', alpha=0.7, color='green')
ax.errorbar(x, ols_coefs, yerr=1.96*ols_se,
            fmt='o', markersize=10, capsize=5, label='OLS', color='blue')
ax.errorbar(x + width, wls_coefs, yerr=1.96*wls_se,
            fmt='s', markersize=10, capsize=5, label='WLS', color='red')

ax.set_ylabel('Coefficient Value')
ax.set_title('OLS vs WLS: Coefficient Estimates with 95% CIs\n(WLS has narrower CIs = more efficient)')
ax.set_xticks(x)
ax.set_xticklabels(coef_names)
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. WLS Diagnostics

Check if WLS corrected the heteroscedasticity:

In [ ]:
# Get WLS residuals
diagnostics_wls = glm_diagnostics(result_wls)
residuals_wls = diagnostics_wls.response_residuals
fitted_wls = result_wls.predict(X)

# Weighted residuals (should have constant variance)
weighted_residuals_wls = residuals_wls * np.sqrt(weights)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# WLS: Weighted residuals vs fitted
axes[0].scatter(fitted_wls, weighted_residuals_wls, alpha=0.6, edgecolor='k', linewidth=0.5, color='green')
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Weighted Residuals')
axes[0].set_title('WLS: Weighted Residuals vs Fitted\n✓ No funnel shape!')
axes[0].grid(alpha=0.3)

# Comparison: OLS vs WLS residuals
axes[1].scatter(df['advertising'], residuals_ols, alpha=0.5, label='OLS (raw)', s=30)
axes[1].scatter(df['advertising'], weighted_residuals_wls, alpha=0.5, label='WLS (weighted)', s=30, color='green')
axes[1].axhline(0, color='r', linestyle='--')
axes[1].set_xlabel('Advertising Spend')
axes[1].set_ylabel('Residuals')
axes[1].set_title('OLS vs WLS Residuals\n(WLS corrects the variance heterogeneity)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Test homoscedasticity on weighted residuals
resid_sq_wls = weighted_residuals_wls**2
aux_result_wls = fit_glm(X=X_test_full, y=resid_sq_wls, family='gaussian')

# Calculate R² manually
y_pred_aux_wls = aux_result_wls.predict(X_test_full)
sse_wls = np.sum((resid_sq_wls - y_pred_aux_wls)**2)
sst_wls = np.sum((resid_sq_wls - np.mean(resid_sq_wls))**2)
r2_aux_wls = 1 - (sse_wls / sst_wls) if sst_wls > 0 else 0

bp_stat_wls = (n - X_test_full.shape[1]) * r2_aux_wls
bp_pval_wls = 1 - stats.chi2.cdf(bp_stat_wls, df=X_test.shape[1])

print(f"\nBreusch-Pagan Test on WLS Weighted Residuals:")
print(f"  Test statistic: {bp_stat_wls:.4f}")
print(f"  p-value: {bp_pval_wls:.4f}")
print(f"\n  Conclusion: {'Heteroscedasticity still present' if bp_pval_wls < 0.05 else 'Homoscedasticity achieved!'}")
print(f"WLS successfully corrected the variance problem!")

## 10. Prediction Comparison

In [ ]:
# Make predictions
y_pred_ols = result_ols.predict(X)
y_pred_wls = result_wls.predict(X)

# Calculate weighted RMSE (WLS should be better at this)
rmse_ols = np.sqrt(np.mean((y - y_pred_ols)**2))
rmse_wls = np.sqrt(np.mean((y - y_pred_wls)**2))
weighted_rmse_ols = np.sqrt(np.mean(weights * (y - y_pred_ols)**2))
weighted_rmse_wls = np.sqrt(np.mean(weights * (y - y_pred_wls)**2))

print("\nPrediction Quality:")
print(f"{'Method':<10} {'RMSE':<10} {'Weighted RMSE':<15}")
print("="*35)
print(f"{'OLS':<10} {rmse_ols:<10.2f} {weighted_rmse_ols:<15.2f}")
print(f"{'WLS':<10} {rmse_wls:<10.2f} {weighted_rmse_wls:<15.2f}")
print("\nWLS has better weighted RMSE (appropriately weighted by precision)")

# Visualize predictions
plt.figure(figsize=(10, 6))
plt.scatter(df['advertising'], df['revenue'], alpha=0.4, s=30, label='Data', color='gray')
plt.plot(df['advertising'], df['true_revenue'], 'g-', lw=3, label='True relationship', zorder=5)

# Sort for line plots
sort_idx = np.argsort(df['advertising'])
plt.plot(df['advertising'].values[sort_idx], y_pred_ols[sort_idx], 
         'b--', lw=2, label='OLS fit', alpha=0.7)
plt.plot(df['advertising'].values[sort_idx], y_pred_wls[sort_idx], 
         'r-', lw=2, label='WLS fit', alpha=0.7)

plt.xlabel('Advertising Spend ($1000s)')
plt.ylabel('Revenue ($1000s)')
plt.title('OLS vs WLS Predictions')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. When to Use WLS?

**Use WLS when:**
1. Residual plots show funnel/cone shape
2. Breusch-Pagan test rejects homoscedasticity
3. You know variance structure from domain knowledge
4. Data has grouped structure with different group sizes

**How to determine weights:**
1. **Known variance**: If Var(εᵢ) = σᵢ², use wᵢ = 1/σᵢ²
2. **Variance function**: If Var(εᵢ) = σ²·xᵢᵖ, use wᵢ = 1/xᵢᵖ
3. **Replicate observations**: Use wᵢ = nᵢ (number of replicates)
4. **Two-stage estimation**: 
   - Fit OLS, get residuals
   - Model |residuals| vs predictors
   - Use fitted values as estimated σᵢ

**Practical tip:** When in doubt, use robust standard errors instead of WLS!

## Key Takeaways

- **Always check** residual plots after fitting
- **WLS ≠ better predictions**, but more efficient estimates
- **Weights matter**: Wrong weights can make things worse!
- **Alternative**: Use robust/sandwich standard errors

## Next Steps

- **Non-linear relationships:** See `01_regression/03_polynomial_vs_gam.ipynb`
- **Robust methods:** See `05_advanced_topics/02_robust_estimation.ipynb`
- **Count data:** See `03_count_data/01_poisson_regression.ipynb` (WLS for overdispersion)

## Resources

- [Aurora-GLM Documentation](https://github.com/Matcraft94/Aurora-GLM)
- [Weighted Least Squares (Penn State)](https://online.stat.psu.edu/stat501/lesson/13)
- [Breusch-Pagan Test](https://en.wikipedia.org/wiki/Breusch%E2%80%93Pagan_test)